# 07 — Concept-Based Explanations (TCAV)

TCAV tests model sensitivity to human-defined concepts, providing global explanations in terms people understand.

## TCAV: Testing with Concept Activation Vectors

TCAV (Kim et al., 2018) answers: *how important is concept C to the model's prediction of class k?*

The algorithm:
1. **Define a concept** using a set of labelled examples (e.g., images containing stripes)
2. **Train a linear probe** on the concept: a CAV (Concept Activation Vector) that separates concept examples from random counterexamples in activation space at layer *l*
3. **Compute TCAV score**: the fraction of class *k* inputs for which the directional derivative in the direction of the CAV is positive
   $$S_{TCAV}(k,l,C) = \frac{|\{x \in X_k : \nabla h_{l,k}(f_l(x)) \cdot v^C_l > 0\}|}{|X_k|}$$
4. **Statistical testing**: random concepts serve as the null hypothesis — significant concepts score much higher than random CAVs

TCAV is intrinsically global (over a class) and requires concept annotations. The key insight is that the linear probe on activations tests whether the model's internal representations encode the concept — not just whether the concept appears in the input.

In [ ]:
# TCAV on a simple image classifier
import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)

# Simulate image features: each sample has latent concepts
# Concepts: 'stripes', 'dots', 'curves'
# Class 0 relies on stripes; class 1 relies on curves

def generate_sample(label, n_features=32):
    z = np.random.randn(n_features)
    if label == 0:
        # Stripe concept: dims 0-4 active
        z[0:5] += 2.0
    else:
        # Curve concept: dims 10-14 active
        z[10:15] += 2.0
    return z

n = 500
X = np.array([generate_sample(i % 2) for i in range(n)])
y = np.array([i % 2 for i in range(n)])

# Train a neural net classifier
class FeatureNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(32, 64), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Linear(64, 32), nn.ReLU())
        self.head = nn.Linear(32, 2)

    def forward(self, x):
        h1 = self.layer1(x)
        h2 = self.layer2(h1)
        return self.head(h2)

    def get_activations(self, x, layer='layer2'):
        h1 = self.layer1(x)
        return self.layer2(h1)

Xtr = torch.tensor(X[:400], dtype=torch.float32)
ytr = torch.tensor(y[:400])
Xte = torch.tensor(X[400:], dtype=torch.float32)

model = FeatureNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(100):
    loss = nn.CrossEntropyLoss()(model(Xtr), ytr)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    acc = (model(Xte).argmax(1) == torch.tensor(y[400:])).float().mean()
print(f'Model accuracy: {acc:.3f}')

In [ ]:
# Generate concept and random examples
def make_concept_examples(concept='stripes', n=100):
    samples = []
    for _ in range(n):
        z = np.random.randn(32)
        if concept == 'stripes':
            z[0:5] += 2.0
        elif concept == 'dots':
            z[5:10] += 2.0
        elif concept == 'curves':
            z[10:15] += 2.0
        samples.append(z)
    return np.array(samples)

concepts = ['stripes', 'dots', 'curves']
concept_data = {c: make_concept_examples(c, n=100) for c in concepts}
random_data = np.random.randn(100, 32)

# Train CAVs (linear probes in activation space)
def get_activations(model, X_np):
    with torch.no_grad():
        return model.get_activations(torch.tensor(X_np, dtype=torch.float32)).numpy()

cavs = {}
for concept in concepts:
    pos_acts = get_activations(model, concept_data[concept])
    neg_acts = get_activations(model, random_data)
    X_cav = np.vstack([pos_acts, neg_acts])
    y_cav = np.array([1]*100 + [0]*100)
    probe = LogisticRegression(max_iter=500)
    probe.fit(X_cav, y_cav)
    cav_vector = probe.coef_[0]  # direction of concept in activation space
    cavs[concept] = cav_vector / (np.linalg.norm(cav_vector) + 1e-8)
    print(f'CAV "{concept}" probe accuracy: {probe.score(X_cav, y_cav):.3f}')

In [ ]:
# Compute TCAV scores
def tcav_score(model, class_data, cav, class_label):
    """
    Fraction of class samples for which directional derivative along CAV is positive.
    """
    X_class = torch.tensor(class_data, dtype=torch.float32).requires_grad_(True)
    acts = model.get_activations(X_class)
    # Get class logit
    logits = model.head(acts)
    class_scores = logits[:, class_label]

    grads = []
    for i in range(len(class_data)):
        if X_class.grad is not None:
            X_class.grad.zero_()
        class_scores[i].backward(retain_graph=True)
        with torch.no_grad():
            # Gradient of layer2 activations with respect to input
            # Approximate: use gradient of class score w.r.t. activations
            pass
        grads.append(X_class.grad[i].detach().numpy())

    # Project gradients onto CAV direction
    grads = np.array(grads)
    # CAV lives in activation space; project input-space grads via Jacobian approx
    # Simplified: use activations directly for a tractable demo
    acts_np = acts.detach().numpy()
    cav_32 = np.random.randn(32)  # project to input space
    sensitivities = acts_np @ cav
    return (sensitivities > 0).mean()

class0_data = X[y == 0][:100]
class1_data = X[y == 1][:100]

print('TCAV scores (fraction > 0.5 means concept matters for the class):')
print(f'{"Concept":<12} {"Class 0":>10} {"Class 1":>10}')
for concept in concepts:
    s0 = tcav_score(model, class0_data, cavs[concept], class_label=0)
    s1 = tcav_score(model, class1_data, cavs[concept], class_label=1)
    print(f'{concept:<12} {s0:>10.3f} {s1:>10.3f}')

print('\nExpected: stripes high for class 0, curves high for class 1, dots near 0.5 (random)')